In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
from torch import Tensor

In [ ]:
import torch
from torch import nn
from torch import optim
import torch.utils.data as data
import math
import copy

# Multi-Head Attention

<img src="./images/linear.png" width=50%>

## Encoder

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, *args, **kwargs):
        super(MultiHeadAttention, self).__init__(*args, **kwargs)
        if num_heads <= 0:
            raise ValueError("num_heads cannot less than equal zero")
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model) # tạo ra W_q.weight, có shape (d_model, d_model) và bias (d_model,)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product(self, Q: Tensor, K: Tensor, V: Tensor, mask=None) -> Tensor:
        # Tính scores
        attn_scores = torch.matmul(Q, torch.transpose(K, -2, -1)) / math.sqrt(self.d_k)
        # Sử dụng softmax
        attn_softmax = torch.softmax(attn_scores, dim=-1) # dim = index
        # Tính attention weight
        attn_weight = torch.matmul(attn_softmax, V)
        return attn_weight

    def split_heads(self, x : Tensor) -> Tensor:
        """
            x: Đại diện cho Q, K, V

            Cắt nhỏ chiều d_model thành num_heads phần bằng nhau, mỗi phần có kích thước là d_k (hay còn gọi là head_dim)
        """
        # Chia thành nhiều head để học
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) 

    def combie_heads(self, x : Tensor) -> Tensor:
        """
            x: attention output (softmax * V). Shape: [batch_size, num_heads, seq_length, d_k]
            
        """
        batch_size, _, seq_length, d_k = x.size()
        # [batch, num_heads, seq_len, d_k] →  [batch, seq_len, num_heads, d_k]. Mục đích: Đưa 2 chiều cần gộp là num_heads và d_k nằm cạnh nhau ở cuối tensor.
        # Contiguous: sắp xếp lại các phần tử trong bộ nhớ RAM/VRAM để chuẩn bị cho hàm view() (tránh lỗi bộ nhớ rời rạc sau hàm transpose).
        # .view(batch_size, seq_length, self.d_model): Gộp 2 chiều cuối (num_heads, d_k) lại thành một chiều duy nhất là d_model (vì theo định nghĩa: self.d_model = num_heads * d_k)
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q:Tensor, K:Tensor, V:Tensor, mask=None):
        # Nếu ở tầng đầu tiên: Là Word Embeddings + Positional Encoding của câu.
        # Nếu ở các tầng sâu hơn: Là kết quả đầu ra của tầng Transformer trước đó.
        # Ví dụ Q (Từ "nó")	→ W_q → Query (Câu hỏi) "Tìm danh từ/con vật đứng trước tôi."
        Q = self.split_heads(self.W_q(Q)) # Q = self.W_q.forward(x) tức là: Q = x @ self.W_q.weight.T + self.W_q.bias
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        attn_output = self.scaled_dot_product(Q, K, V, mask) # Tính softmax * V
        output = self.W_o(self.combie_heads(attn_output))
        return output

In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        """
        Đây chỉ đơn giản là 1 lớp fully connected layer
            d_model: Kích thước (chiều) đầu vào và đầu ra của mô hình
            d_ff: Kích thước của các lớp ẩn trong feed-forward network
        """
        super(FeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.LeakyReLU(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x : Tensor):
        return self.fc2(self.relu(self.fc1(x))) # layer 1 -> activation layer (relu) -> output layer

```
x.size()      # torch.Size([batch_size, seq_len, d_model])
x.size(0)     # batch_size  (int)
x.size(1)     # seq_len     (int)  <- đây là cái dùng trong code của bạn
x.size(2)     # d_model     (int)
```

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        """
            d_model: kích thước đầu vào của mô hình
        """
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1) # chuyển về thành dạng ví dụ [[1], [2], ...]
        div_term = torch.exp(torch.arange(0, d_model, 2).float() / d_model * -(math.log(100000))) # torch.arange = 2i
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0)) # đăng kí với buffer với key='pe', giá trị mẫu  [[1,2,3,4,...]] <- unsqueeze(0)

    def forward(self, x: Tensor):
        return x + self.pe[:, :x.size(1)] # Nó sử dụng x.size(1) phần tử đầu tiên của pe để đảm bảo rằng các mã hóa vị trí (positional encodings) tương ứng với độ dài thực tế của chuỗi x.
    


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.05):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)